# Twin Delayed DDPG (TD3)

https://spinningup.openai.com/en/latest/algorithms/td3.html

In [1]:
import torch
from tensordict.nn import TensorDictModule, TensorDictSequential
from torch import nn
from torchrl import logger
from torchrl.collectors import Collector
from torchrl.data import LazyTensorStorage, ReplayBuffer
from torchrl.envs import Compose, DoubleToFloat, GymEnv, StepCounter, TransformedEnv
from torchrl.modules import (
    AdditiveGaussianModule,
    ProbabilisticActor,
    TanhDelta,
    ValueOperator,
)
from torchrl.objectives import TD3Loss, SoftUpdate

/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:574: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  PUCT = functools.partial(PUCTScore, c=5)  # AlphaGo default value
/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:575: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB = functools.partial(UCBScore, c=math.sqrt(2))  # default from Auer et al. 2002
/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:576: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB1_TUNED = functools.partial(
/usr/local/Caskroom/mini

In [2]:
env = TransformedEnv(
    GymEnv("InvertedPendulum-v5"),
    Compose([DoubleToFloat(), StepCounter()]),
)

In [3]:
_ = env.set_seed(0)
_ = torch.manual_seed(0)

Use the same policy as in DDPG.
The clipped noise is added (to the action) by the loss module when computing the actor loss.

In [4]:
NUM_CELLS = 256

policy_net = nn.Sequential(
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(1),
)

policy_module = ProbabilisticActor(
    TensorDictModule(policy_net, in_keys=["observation"], out_keys=["param"]),
    in_keys=["param"],
    spec=env.action_spec,
    safe=True,
    distribution_class=TanhDelta,
    distribution_kwargs={
        "low": env.action_spec.space.low,  # type: ignore
        "high": env.action_spec.space.high,  # type: ignore
    },
)

Use the same Q-value network as in DDPG;
the network is duplicated by the loss module.

In [5]:
class ActionValueNetwork(nn.Module):
    def __init__(self, observation_net: nn.Module, action_net: nn.Module):
        super().__init__()
        self.observation_net = observation_net
        self.action_net = action_net

    def forward(self, observation: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        xs = self.observation_net(observation)
        xs = nn.Tanh()(xs)
        xs = torch.cat([xs, action], dim=-1)
        xs = self.action_net(xs)

        return xs


value_net = ActionValueNetwork(
    observation_net=nn.Sequential(
        nn.LazyLinear(NUM_CELLS), nn.Tanh(), nn.LazyLinear(NUM_CELLS)
    ),
    action_net=nn.Sequential(nn.LazyLinear(NUM_CELLS), nn.Tanh(), nn.LazyLinear(1)),
)

value_module = ValueOperator(value_net, in_keys=["observation", "action"])

In [6]:
_ = value_module(env.rollout(10, policy_module))  # initialize lazy layers

In [7]:
LEARNING_RATE = 1e-4

loss_module = TD3Loss(
    actor_network=policy_module,
    qvalue_network=value_module,
    action_spec=env.action_spec,
    num_qvalue_nets=2,  # use two Q-value networks (original "twin")
    policy_noise=0.2,  # std deviation of policy noise
    noise_clip=0.5,
)

updater = SoftUpdate(loss_module, eps=0.99)

optimizer = torch.optim.Adam(loss_module.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 100_000)

Create a stochastic policy by adding Gaussian noise to the output of the
(deterministic) policy to ensure a certain level of initial exploration.

In [8]:
exploration_module = AdditiveGaussianModule(
    spec=env.action_spec,
    annealing_num_steps=100_000,
    safe=True,
)

exploration_policy = TensorDictSequential([policy_module, exploration_module])

The collector takes random actions for a fixed number of steps (on top of using the exploration policy) 
to ensure that the data is varied enough.

Use a large replay buffer for off-policy algorithms since the policy used to obtain experiences is irrelevant;
the Bellman equation should be satisfied for all transitions.

In [9]:
FRAMES_PER_BATCH = 100
INIT_RANDOM_FRAMES = 5000

collector = Collector(
    env,
    policy=exploration_policy,
    frames_per_batch=FRAMES_PER_BATCH,
    init_random_frames=INIT_RANDOM_FRAMES,
    total_frames=-1,
)

buffer = ReplayBuffer(storage=LazyTensorStorage(max_size=100_000))

Same training loops as in DQN and DDPG, with the exception that the target is updated less frequently ("delayed" DDPG).

In [10]:
OPTIM_STEPS = 10
BATCH_SIZE = 128

step_count = 0
episode_count = 0


for idx, data in enumerate(collector, start=1):
    buffer.extend(data)

    step_count += data.numel()
    episode_count += data["next", "done"].sum()

    if len(buffer) < collector.init_random_frames:
        continue

    max_steps = data["next", "step_count"].max()

    if idx % 10 == 0:
        logger.info(f"[{idx:>3}] steps: {max_steps:>3}")

    if max_steps > 200:
        break

    for optim_step in range(OPTIM_STEPS):
        data_batch = buffer.sample(BATCH_SIZE)
        batch_loss = loss_module(data_batch)

        loss = batch_loss["loss_actor"] + batch_loss["loss_qvalue"]
        loss.backward()

        nn.utils.clip_grad_norm_(loss_module.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

        # update the target once per two update of the Q-network
        if optim_step % 2 == 0:
            updater.step()

        scheduler.step()

    exploration_module.step(data.numel())

logger.info(f"solved after {step_count} steps, {episode_count} episodes")

2026-02-16 10:51:10,631 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([100000]) shape [END]
2026-02-16 10:51:16,236 [torchrl][INFO]    [ 50] steps:  15 [END]
2026-02-16 10:51:20,679 [torchrl][INFO]    [ 60] steps:   7 [END]
2026-02-16 10:51:24,406 [torchrl][INFO]    [ 70] steps:   6 [END]
2026-02-16 10:51:28,139 [torchrl][INFO]    [ 80] steps:   9 [END]
2026-02-16 10:51:32,004 [torchrl][INFO]    [ 90] steps:  37 [END]
2026-02-16 10:51:36,132 [torchrl][INFO]    [100] steps:  59 [END]
2026-02-16 10:51:40,707 [torchrl][INFO]    [110] steps:  26 [END]
2026-02-16 10:51:44,907 [torchrl][INFO]    [120] steps:  40 [END]
2026-02-16 10:51:49,102 [torchrl][INFO]    [130] steps:  67 [END]
2026-02-16 10:51:53,046 [torchrl][INFO]    [140] steps:  79 [END]
2026-02-16 10:51:57,080 [torchrl][INFO]    [150] steps:  44 [END]
2026-02-16 10:52:01,117 [torchrl][INFO]    [160] steps:  79 [END]
2026-02-16 10:52:05,283 [torchrl][INFO]    [170] steps:  82 [END]
2026-02-16 10:52:09,247 [torchr